## Imports

In [2]:
## Imports
import os
import sys
import json
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

# Se placer à la racine du projet
project_root = Path.cwd()

if not (project_root / "api").exists():
    project_root = project_root.parent

os.chdir(project_root)
sys.path.insert(0, str(project_root))

load_dotenv()

from api.main import apply_feature_engineering

c:\Users\ethan\Documents\OpenClassrooms\MLops_2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



## 1. Chargement des données de référence

In [3]:
chemin_reference = "data/reference_data.csv"
reference_data = pd.read_csv(chemin_reference)


## 2. Chargement des données de production (logs de l'API)

In [4]:
DATABASE_URL = os.getenv("DATABASE_URL")

if DATABASE_URL and DATABASE_URL.startswith("postgres://"):
    DATABASE_URL = DATABASE_URL.replace("postgres://", "postgresql://", 1)

if not DATABASE_URL:
    raise ValueError("La variable DATABASE_URL n'est pas configurée dans le fichier .env")

engine = create_engine(DATABASE_URL)
query = "SELECT input_data FROM prediction_logs WHERE status = 'success'"
df_logs = pd.read_sql(query, engine)

if df_logs.empty:
    print("La base de données ne contient aucun log d'inférence en succès.")
    production_data = pd.DataFrame(columns=reference_data.columns)
else:
    production_data = pd.json_normalize(df_logs['input_data'])
    print(f"Données de production chargées depuis Neon : {production_data.shape[0]} lignes.")

Données de production chargées depuis Neon : 2 lignes.



## 3. Application du même feature engineering que l'API


In [5]:
with open("api/expected_features.json", "r") as f:
    expected_features = json.load(f)

reference_engineered = apply_feature_engineering(reference_data)
production_engineered = apply_feature_engineering(production_data)

reference_data_aligned = reference_engineered.reindex(
    columns=expected_features,
    fill_value=0
)

production_data_aligned = production_engineered.reindex(
    columns=expected_features,
    fill_value=0
)

## 4. Initialisation du rapport avec les métriques de Data Drift

In [6]:
drift_report = Report(metrics=[DataDriftPreset()])



## 5. Calcul des statistiques comparatives (Référence vs Production)


In [7]:
if not production_data_aligned.empty:
    # Lancement des calculs
    drift_report.run(reference_data=reference_data_aligned, current_data=production_data_aligned)
    
    # Affichage du rapport directement sous la cellule
    drift_report.show(mode='inline')
else:
    print("Calcul impossible : le jeu de données de production est vide.")

c:\Users\ethan\Documents\OpenClassrooms\MLops_2\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\ethan\Documents\OpenClassrooms\MLops_2\.venv\Lib\site-packages\scipy\stats\_stats_py.py:7192: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs - f_exp)**2 / f_exp
c:\Users\ethan\Documents\OpenClassrooms\MLops_2\.venv\Lib\site-packages\scipy\stats\_stats_py.py:7192: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs - f_exp)**2 / f_exp
c:\Users\ethan\Documents\OpenClassrooms\MLops_2\.venv\Lib\site-packages\scipy\stats\_stats_py.py:7192: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs - f_exp)**2 / f_exp
c:\Users\ethan\Documents\OpenClassrooms\MLops_2\.venv\Lib\site-packages\scipy\stats\_stats_py.py:7192: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs - f_exp)**2 / f_exp
c:\Users\ethan\Documents\OpenClassrooms\ML


## 6. Sauvegarde du rapport interactif


In [8]:
if not production_data_aligned.empty:
    os.makedirs("reports", exist_ok=True)

    chemin_rapport = "reports/data_drift_report.html"
    drift_report.save_html(chemin_rapport)

    print(f"Rapport généré avec succès et sauvegardé sous : {chemin_rapport}")
else:
    print("Aucun rapport à sauvegarder.")

Rapport généré avec succès et sauvegardé sous : reports/data_drift_report.html


## Conclusion

Ce notebook simule une étape de monitoring MLOps en comparant les données de référence issues de l'entraînement avec les données reçues par l'API en production.

Les données de production sont récupérées depuis les logs d'inférence stockés en base PostgreSQL/Neon. Afin de surveiller les données réellement utilisées par le modèle, le même feature engineering que celui de l'API est appliqué aux données de référence et aux données de production.

Le rapport Evidently permet ensuite d'identifier d'éventuelles dérives dans les distributions des features finales envoyées au modèle. Cette analyse ne prouve pas à elle seule une dégradation du modèle, mais elle démontre la mise en place d'un mécanisme de surveillance capable d'alerter lorsqu'un écart apparaît entre les données d'entraînement et les données réellement traitées en production.